# **RSM434 ALGO2e v3: forecast-driven market maker**

The strategy lives in `algo2e.py`; this notebook only runs it. v2 is kept in git history and frozen in `research/v2_strategy.py` for comparison.

**What the data says.** `recorder.py` logged the practice server (books + time & sales, 8 snapshots/s) for 8 cases (one joined mid-case) with 5–15 rival algos in the top of the book. The three stocks behave differently:

| | variance ratio at 10 s (1 = random walk) | best out-of-sample forecast of the mid | OOS R² |
|---|---|---|---|
| RY  | **0.20–0.30 in all 8 cases** (snaps back) | 2.4·(EMA4 − mid) − 1.3·(EMA8 − mid), 3 s ahead | **0.20** |
| CNR | 0.3–2.0: trends when the room is crowded, bounces when it is quiet | −1.6·(EMA1 − mid) + 1.6·imbalance, 3 s ahead | 0.09 |
| AC  | 0.5–2.5, same pattern | −0.5·(EMA1 − mid), 1 s ahead | 0.12 (none at 3 s) |

v2 centred its quotes on the current microprice, so after every RY spike it re-centred at the top and bought there. In crowded rooms the median CNR and AC spread is 1–2 ticks and market makers lose to the trends: v2's past reports show it, and so does the replay.

**What v3 does**
* **Quote around a forecast.** Reservation price = mid + forecast − skew·position/cap. Each rung of the ladder sits at the most aggressive price whose expected edge (reservation vs. price, plus the passive rebate) clears that rung's minimum. When the forecast beats the far side by the taker fee + a margin, cross (RY *pays* 0.14 ticks to take).
* **Put the budget where the edge is.** RY gets 20,000 of the 24,000-share budget and a 4-deep ladder (edges 0.8 / 2 / 3.5 / 5.5 ticks, 5,000 each), which is filled by the sweeps that revert. CNR gets 4,000. AC is off: it lost in every crowded case. Moving budget to CNR/AC when their spreads widen (`WIDE`) was tested and lost money in 7 of 8 cases (the 8th tied), so it is off.
* **Check the forecast live.** Every forecast is scored against the mid `h` seconds later; its weight is the realized slope, shrunk toward 1 and clipped to [0, 1.5]. If the forecast stops working, it fades out and the algo is a plain skewed market maker.
* **No limit fines, by construction.** Open orders are read *before* positions, so every fill is in one or the other. Every resting share counts against the cap, a replacement waits until the next position read (`STRICT`), and caps sum to 24,000 (1,000 under the limit). v2's fines came from replacements going out while the old order could still fill, and from reading positions and orders in parallel.
* **Stay under 100 requests/s.** RIT rejects anything over 100 commands/second per trader. A token bucket caps us at 92/s, so 429s are rare and the loop never stalls; v2 lost up to 87 % of its cycles to them. A ticker whose book read fails just sits out that cycle.
* **Every run is recorded** to `runs/` (same format the backtest reads), so the forecast can be refitted on real competition data.

**Evidence (replay of 8 recorded cases, one partial, queue-position fills, orders one snapshot late)**: v2 averages +$68K per case, v3 +$218K. In the 4 most crowded cases (11–15 rivals at the touch): v2 +$8K, v3 +$148K, with v2 losing on CNR and AC in all four. v3 never went over 24,000 gross; v2 broke the 25,000 gross limit in 7 of 8. End to end against `research/mock_rit.py` at real speed, `algo2e.main()` made +$224K on a case the backtest puts at $188K–$245K, with 0 fines and 3 HTTP 429s in 5 minutes. The replay cannot show how rivals react to our quotes, so expect less live.

**Research tools** (`research/`): `analyze.py` (predictability), `fit.py` (out-of-sample forecast fit), `backtest.py` (v2 vs v3 replay), `tune.py` / `sweep.py` / `robust.py` (parameter checks), `mock_rit.py` + `e2e.py` (run the live loop against a local RIT stand-in).

# **Settings**
Everything is in `algo2e.py` (credentials, `CFG` per ticker, risk and rate settings). Override here for a session if you like; re-running this cell reloads the file.

In [ ]:
import importlib

import algo2e
importlib.reload(algo2e)

# algo2e.BASE_URL = 'http://flserver.rotman.utoronto.ca:16585/v1'
# algo2e.CFG['RY']['cap'] = 16000        # e.g. trade smaller
print({t: (c['cap'], [r[0] for r in c['rungs']]) for t, c in algo2e.CFG.items()}, 'budget', algo2e.BUDGET)

# **Run**
Start it before or during the case; it waits for the case to go live, trades until it stops, cancels everything, then prints and saves the report under `diagnostics/`.

In [ ]:
try:
    algo2e.main()
except KeyboardInterrupt:
    algo2e.api('POST', 'commands/cancel', timeout=5, retry=3, all=1)
    print('Interrupted, all orders cancelled.')

In [ ]:
# Read-only. Pulls the server's recap again (RIT deletes fills and transactions when the next case starts).
algo2e.report()

# **Record a practice case (read-only)**
Sends no orders. Writes `data/*.jsonl.gz`, which `research/fit.py` and `research/backtest.py` read. Run it while someone else's case is live to collect more data.

In [ ]:
import recorder
recorder.record(330)   # seconds